# Limb vs neuron OpenSpliceAI delta scores
对 limb 和 neuron VCF 的 OpenSpliceAI 四个 delta score (DS_AG, DS_AL, DS_DG, DS_DL) 做交集比较，并用表型信息标色（Abnormality_of_limbs / Abnormality_of_the_nervous_system）。同时筛选 splice_donor_variant_Likely_LOF / splice_acceptor_variant_Likely_LOF 类型单独出图，四种表型组合固定配色。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D
from pathlib import Path

sns.set(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120

In [ ]:
limb_path = Path("/home1/xyf/data/openspliceai_tissue_data/variant/limb_400_train.vcf")
neuron_path = Path("/home1/xyf/data/openspliceai_tissue_data/variant/neuron_400_train.vcf")

if not limb_path.exists() or not neuron_path.exists():
    raise FileNotFoundError("请确认 VCF 路径是否存在")

In [ ]:
# 解析 INFO 字段、OpenSpliceAI 分数和表型归类

def parse_info_field(info_str):
    if pd.isna(info_str):
        return {}
    info = {}
    for entry in str(info_str).split(";"):
        if "=" in entry:
            key, value = entry.split("=", 1)
            info[key] = value
    return info


def parse_openspliceai(value):
    if pd.isna(value):
        return (None, None, None, None)
    parts = str(value).split("|")
    if len(parts) < 6:
        return (None, None, None, None)
    try:
        return tuple(float(parts[i]) if parts[i] != "" else None for i in range(2, 6))
    except ValueError:
        return (None, None, None, None)


target_terms = {
    "limbs": ["abnormality_of_limbs"],
    "nervous": ["abnormality_of_the_nervous_system", "abnormality_of_nervous_system"],
}

phenotype_order = ["neither", "limbs_only", "nervous_only", "both"]
legend_labels = {
    "neither": "Neither",
    "limbs_only": "Abnormality_of_limbs",
    "nervous_only": "Abnormality_of_the_nervous_system",
    "both": "Both",
}


def normalize_pheno(text: str) -> str:
    normalized = str(text).replace(" ", "_").lower()
    return normalized


def assign_phenotype_group(text: str) -> str:
    normalized = normalize_pheno(text)
    has_limb = any(term in normalized for term in target_terms["limbs"])
    has_neuro = any(term in normalized for term in target_terms["nervous"])
    if has_limb and has_neuro:
        return "both"
    if has_limb:
        return "limbs_only"
    if has_neuro:
        return "nervous_only"
    return "neither"


def load_vcf(path: Path) -> pd.DataFrame:
    cols = ["CHROM", "POS", "ID", "REF", "ALT", "QUAL", "FILTER", "INFO"]
    df = pd.read_csv(
        path,
        sep="	",
        comment="#",
        header=None,
        names=cols,
        dtype={"CHROM": str, "POS": int, "ID": str, "REF": str, "ALT": str, "QUAL": str, "FILTER": str, "INFO": str},
    )

    info_expanded = df["INFO"].apply(parse_info_field).apply(pd.Series)
    df = pd.concat([df.drop(columns=["INFO"]), info_expanded], axis=1)

    df[["ds_ag", "ds_al", "ds_dg", "ds_dl"]] = df["OpenSpliceAI"].apply(parse_openspliceai).apply(pd.Series)
    df["phenotypes_raw"] = df["PHENOTYPES"].fillna("")
    df["phenotype_group"] = df["phenotypes_raw"].apply(assign_phenotype_group)
    df["phenotype_group"] = pd.Categorical(df["phenotype_group"], categories=phenotype_order)
    df["variant_key"] = df.apply(lambda r: f"{r['CHROM']}:{r['POS']}_{r['REF']}>{r['ALT']}", axis=1)
    return df


In [ ]:
limb_df = load_vcf(limb_path)
neuron_df = load_vcf(neuron_path)

print(f"Limb variants: {len(limb_df)} | Neuron variants: {len(neuron_df)}")

merged = limb_df.merge(
    neuron_df,
    on=["variant_key", "CHROM", "POS", "REF", "ALT"],
    suffixes=("_limb", "_neuron"),
)

merged["phenotypes_combined"] = merged[["phenotypes_raw_limb", "phenotypes_raw_neuron"]].fillna("").agg("|".join)
merged["phenotype_group"] = merged["phenotypes_combined"].apply(assign_phenotype_group)
merged["phenotype_group"] = pd.Categorical(merged["phenotype_group"], categories=phenotype_order)

merged["consequence_combined"] = (
    merged["CONSEQUENCE_limb"].fillna("").str.lower()
    + "|"
    + merged["CONSEQUENCE_neuron"].fillna("").str.lower()
)

delta_cols = ["ds_ag", "ds_al", "ds_dg", "ds_dl"]
for col in delta_cols:
    merged = merged.dropna(subset=[f"{col}_limb", f"{col}_neuron"])

print(f"Merged variants (intersection): {len(merged)}")

In [ ]:
palette = {
    "neither": "#9ca3af",
    "limbs_only": "#f97316",
    "nervous_only": "#3b82f6",
    "both": "#22c55e",
}

delta_info = [
    ("ds_ag", "Delta score (acceptor gain)"),
    ("ds_al", "Delta score (acceptor loss)"),
    ("ds_dg", "Delta score (donor gain)"),
    ("ds_dl", "Delta score (donor loss)"),
]


def plot_delta_pairs(df: pd.DataFrame, title: str):
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    axes = axes.ravel()

    for ax, (col, desc) in zip(axes, delta_info):
        xcol = f"{col}_limb"
        ycol = f"{col}_neuron"
        subset = df[[xcol, ycol, "phenotype_group"]].dropna()

        if subset.empty:
            ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
            ax.set_title(desc)
            continue

        sns.scatterplot(
            data=subset,
            x=xcol,
            y=ycol,
            hue="phenotype_group",
            hue_order=phenotype_order,
            palette=palette,
            alpha=0.8,
            edgecolor="white",
            linewidth=0.4,
            s=40,
            ax=ax,
            legend=False,
        )

        min_val = subset[[xcol, ycol]].min().min()
        max_val = subset[[xcol, ycol]].max().max()
        padding = max(0.01, (max_val - min_val) * 0.05)
        ax.plot([min_val - padding, max_val + padding], [min_val - padding, max_val + padding], "k--", lw=1)

        ax.set_xlabel(f"Limb {desc}")
        ax.set_ylabel(f"Neuron {desc}")
        ax.set_title(desc)

    legend_handles = [
        Line2D([0], [0], marker='o', linestyle='', markersize=6, color=palette[group], label=legend_labels[group])
        for group in phenotype_order
    ]
    fig.legend(legend_handles, [legend_labels[g] for g in phenotype_order], title="Phenotypes", loc="upper center", ncol=4, frameon=False)
    fig.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()


In [ ]:
plot_delta_pairs(merged, "Limb vs neuron: all variants")

lof_mask = (
    merged["consequence_combined"].str.contains("splice_donor_variant_likely_lof")
    | merged["consequence_combined"].str.contains("splice_acceptor_variant_likely_lof")
)
lof_df = merged[lof_mask].copy()
print(f"splice donor/acceptor Likely_LOF variants: {len(lof_df)}")

if not lof_df.empty:
    plot_delta_pairs(lof_df, "Splice donor/acceptor Likely_LOF variants")
else:
    print("没有满足 splice donor/acceptor Likely_LOF 条件的交集变异。")
